# photoD on DP2

Runs the `fixed_fast` branch over a prepared DP2 catalog. Everything you need to change is in the first cell.

What this does differently from `DP2_run_gpu-2.ipynb`:

1. It does not use `merge_map`. That join keeps one star partition per prior map pixel and drops the rest,
   which on DP2 is fifteen in sixteen, silently. The prior maps are read into one array here and each
   partition looks its sightline up instead.
2. It uses `LSSTlocus_10Gyr_DP2.txt`, whose u-g and main sequence are calibrated on DP2 stars.
3. It adds `colorErrFloor` for the error the locus itself carries, on top of the photometric systematic you
   already put in during preparation.
4. The A_r grid reaches past the extinction of the field instead of stopping at 2.5 mag.
5. It counts the rows in and the rows out and refuses to finish quietly if they differ.

## What to set

In [ ]:
# paths
photodSrc   = "../photoD/src"
locusPath   = "../photoD/data/LSSTlocus_10Gyr_DP2.txt"
dp2Url      = "/mnt/beegfs/scratch/data/tmp_lsst_dp2/dp2_to_run/"
priorMapUrl = "/mnt/beegfs/scratch/data/TRILEGAL-18-06-2026/priors_hats_bin_halo_tLoc/"
priorFile   = "priors_tLoc.npz"          # written once by section 2, reused after that
outPath     = "/mnt/beegfs/scratch/data/photoD_results"
outName     = "dp2-2"

# machine
nGPUs       = 4
nWorkers    = 8          # two per GPU
chunkSize   = 200        # partitions between worker restarts, which is what bounds the memory

# fit
fitColors     = ("ug", "gr", "ri", "iz", "zy")
batchSize     = 400      # 100 to 400 all run at full speed, 10 is thirty times slower, 800 spills
arMax         = 8.0      # top of the A_r grid in magnitudes; the width itself costs nothing
colorErrFloor = 0.03     # the locus is not exact and the fit has no other way to know
arMapColumn   = "Ar_SFD"

In [ ]:
import os

# The autotuner compiles and times dozens of variants of every kernel the first time it meets one, which is
# minutes of CPU per worker with the GPU idle and buys this fit nothing. Both have to be set before JAX starts.
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import sys
import time
from pathlib import Path

import dask
import jax
import lsdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from dask.distributed import Client, get_worker

sys.path.append(photodSrc)

from photod.bayes import getEstimatesMeta, makeBayesEstimates3d
from photod.locus import LSSTsimsLocus, get3DmodelList, make3DlocusList, subsampleLocusData
from photod.parameters import GlobalParams
from photod.priors import getBayesConstants, priorGridFromMaps

assert Path(locusPath).exists(), f"no locus at {locusPath}: is the checkout on fixed_fast and pulled?"
print("jax sees", jax.device_count(), "devices")

## 1. The prior maps, as one array

Her prior maps are a HATS catalog with one row per r bin and sky pixel. Joining stars against it is what
loses partitions, so they are read once into a single array with an index from HEALPix pixel to row. Run this
once; afterwards the file is reused.

In [ ]:
%%time
import cdshealpix
from astropy.coordinates import Latitude, Longitude
from hats.io.paths import pixel_catalog_file

def buildPriorFile(priorMapUrl, priorFile):
    """Every prior map in one array, with an index from HEALPix pixel to its row."""
    catalog = lsdb.open_catalog(priorMapUrl)
    pixels = catalog.hc_structure.get_healpix_pixels()
    base = catalog.hc_structure.catalog_base_dir
    order = max(p.order for p in pixels)
    bc = getBayesConstants()
    rGrid = np.linspace(bc["rmagMin"], bc["rmagMax"], bc["rmagNsteps"])

    cube, rows, index = [], [], np.full(12 * 4 ** order, -1, dtype=np.int32)
    xGrid = yGrid = None
    for pixel in pixels:
        table = pq.read_table(str(pixel_catalog_file(base, pixel))).to_pandas()
        if not len(table):
            continue
        rmag = table["rmag"].to_numpy(dtype=float)
        X = np.frombuffer(table["xGrid"].iloc[0], dtype=np.float64)
        nX = np.unique(X).size
        X = X.reshape(-1, nX)
        Y = np.frombuffer(table["yGrid"].iloc[0], dtype=np.float64).reshape(X.shape)
        maps = np.stack([
            np.frombuffer(table["kde"].iloc[int(np.argmin(np.abs(rmag - r)))], dtype=np.float64).reshape(X.shape)
            for r in rGrid
        ]).astype(np.float32)
        xGrid, yGrid = X[0], Y[:, 0]
        # every child of this pixel at the finest order points at the same row
        spread = 4 ** (order - pixel.order)
        first = pixel.pixel * spread
        index[first:first + spread] = len(cube)
        cube.append(maps)
        rows.append(pixel.pixel)
    np.savez(priorFile, kde=np.stack(cube), rmag=rGrid, xGrid=xGrid, yGrid=yGrid,
             index=index, order=order)
    print(f"{len(cube)} pixels of order {order}, maps {np.stack(cube).shape}, "
          f"{np.stack(cube).nbytes / 2 ** 30:.2f} GiB -> {priorFile}")

if not Path(priorFile).exists():
    buildPriorFile(priorMapUrl, priorFile)
else:
    print(priorFile, "already there, skipping")

## 2. The locus and the fit setup

In [ ]:
%%time
colnames = ["tLoc", "Mr", "FeH", "ug", "gr", "ri", "iz", "zy"]
LSSTlocus = LSSTsimsLocus(fixForStripe82=False, datafile=locusPath, colnames=colnames)
locusData = subsampleLocusData(LSSTlocus, kMr=1, kFeH=1, yLabel="tLoc")
ArGridList, locus3DList = get3DmodelList(locusData, fitColors, yLabel="tLoc")

# the standard grid stops at 2.5 mag, and a star whose A_r is above the top of it has it pinned there, which
# throws the distance out with it. The prior reaches 1.3 A_r(map) + 0.1, so the grid has to clear that.
ArGridList["ArLarge"] = np.arange(0, arMax + 1e-9, 0.02)
locus3DList["ArLarge"] = make3DlocusList(locusData, fitColors, [ArGridList["ArLarge"]], yLabel="tLoc")[0]

globalParams = GlobalParams(
    fitColors, locusData, ArGridList, locus3DList,
    yLabel="tLoc", MrColumn="tLoc", computeMrTrue=True,
    ArMapColumn=arMapColumn, colorErrFloor=colorErrFloor,
)
print("locus", len(locusData), "points, A_r grid", ArGridList["ArLarge"].size, "points")

## 3. The fit

One task per partition file. The task reads its own file, so nothing hands a worker a piece of a catalog and
with it the structure of the whole survey. Workers are replaced every `chunkSize` partitions because reading
leaves memory behind that releasing does not recover.

In [ ]:
STATE = {}

def loadOnce(priorFile, setup):
    """The prior maps and the fit setup, built once per worker process rather than sent to it."""
    if "priors" not in STATE:
        STATE["priors"] = dict(np.load(priorFile))
        locusPath, fitColors, arMax, colorErrFloor, arMapColumn = setup
        locus = LSSTsimsLocus(fixForStripe82=False, datafile=locusPath,
                              colnames=["tLoc", "Mr", "FeH", "ug", "gr", "ri", "iz", "zy"])
        data = subsampleLocusData(locus, kMr=1, kFeH=1, yLabel="tLoc")
        grids, models = get3DmodelList(data, fitColors, yLabel="tLoc")
        grids["ArLarge"] = np.arange(0, arMax + 1e-9, 0.02)
        models["ArLarge"] = make3DlocusList(data, fitColors, [grids["ArLarge"]], yLabel="tLoc")[0]
        STATE["params"] = GlobalParams(fitColors, data, grids, models, yLabel="tLoc", MrColumn="tLoc",
                                       computeMrTrue=True, ArMapColumn=arMapColumn,
                                       colorErrFloor=colorErrFloor)
        try:
            name = int(get_worker().name)
        except Exception:
            name = os.getpid()
        STATE["device"] = jax.devices()[name % jax.device_count()]
    return STATE["priors"], STATE["params"], STATE["device"]


def fitAndWrite(source, order, npix, outBase, priorFile, setup, batchSize):
    """Fit one partition file and write it where its HEALPix pixel belongs; return how many stars it held."""
    from hats.io.paths import pixel_catalog_file
    from hats.pixel_math import HealpixPixel

    stars = pq.read_table(source).to_pandas()
    if not len(stars):
        return 0
    priors, params, device = loadOnce(priorFile, setup)
    pixel = np.asarray(cdshealpix.nested.lonlat_to_healpix(
        Longitude(stars["ra"].to_numpy(), unit="deg"),
        Latitude(stars["dec"].to_numpy(), unit="deg"),
        int(priors["order"])))
    row = priors["index"][pixel]
    pieces = []
    for value in np.unique(row[row >= 0]):
        part = stars[row == value]
        grid = priorGridFromMaps(priors["kde"][value], priors["rmag"],
                                 priors["xGrid"], priors["yGrid"], params)
        with jax.default_device(device):
            estimates, _ = makeBayesEstimates3d(part, jax.numpy.array(list(grid.values())),
                                                params, batchSize=batchSize)
        pieces.append(estimates)
    if not pieces:
        return 0
    out = pd.concat(pieces, ignore_index=True)
    path = pixel_catalog_file(outBase, HealpixPixel(order, npix))
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(path, index=False)
    return len(out)

In [ ]:
%%time
from hats.io.paths import pixel_catalog_file
import shutil

dp2 = lsdb.open_catalog(dp2Url)
dp2Base = dp2.hc_structure.catalog_base_dir
sources = [(str(pixel_catalog_file(dp2Base, p)), p.order, p.pixel)
           for p in dp2.hc_structure.get_healpix_pixels()]
starsIn = sum(pq.ParquetFile(s[0]).metadata.num_rows for s in sources)
print(f"{len(sources)} partitions holding {starsIn:,} stars")

outBase = Path(outPath) / outName
if outBase.exists():
    shutil.rmtree(outBase)
outBase.mkdir(parents=True, exist_ok=True)

setup = (locusPath, fitColors, arMax, colorErrFloor, arMapColumn)
dask.config.set({"distributed.worker.memory.target": False,
                 "distributed.worker.memory.spill": False,
                 "distributed.worker.memory.pause": False})

starsOut, done = 0, 0
start = time.time()
for begin in range(0, len(sources), chunkSize):
    group = sources[begin:begin + chunkSize]
    # a fresh cluster per chunk: reading partitions leaves memory behind that no release recovers
    with Client(n_workers=nWorkers, threads_per_worker=1, dashboard_address=None):
        counts = dask.compute(*[dask.delayed(fitAndWrite)(src, order, npix, outBase, priorFile,
                                                          setup, batchSize)
                                for src, order, npix in group])
    starsOut += int(sum(counts))
    done += len(counts)
    print(f"  {done}/{len(sources)} partitions, {starsOut:,} stars, "
          f"{(time.time() - start) / 60:.1f} min", flush=True)
print(f"done in {(time.time() - start) / 60:.1f} minutes")

## 4. Did every star come out the other side

The check that catches a join quietly dropping partitions. These two numbers have to be equal.

In [ ]:
assert starsOut == starsIn, f"{starsIn:,} stars went in and {starsOut:,} came out: {starsIn - starsOut:,} lost"
print(f"{starsOut:,} stars in and out")

## 5. Write the catalog metadata beside the files

In [ ]:
from hats.catalog import PartitionInfo, TableProperties
from hats.io import write_parquet_metadata
from hats.pixel_math import HealpixPixel

written = [HealpixPixel(order, npix) for _, order, npix in sources
           if pixel_catalog_file(outBase, HealpixPixel(order, npix)).exists()]
PartitionInfo.from_healpix(written).write_to_file(catalog_path=outBase)
TableProperties(catalog_name=outName, catalog_type="object", total_rows=starsOut,
                ra_column="ra", dec_column="dec").to_properties_file(outBase)
write_parquet_metadata(outBase)
print("written", outBase)

## 6. A look at what came out

`flags` is one bit per thing worth knowing: 1 the locus does not pass through the colours or the fit found
nothing, 2 the Mr posterior is lopsided so a giant and a dwarf solution both survived, 4 [Fe/H] is against the
end of the model grid, 8 A_r is against the top of its grid, 16 a colour had no measurement. Nothing is
dropped for being flagged, and `flags & 3 == 0` is the cut to start from.

`DM` is the distance modulus, so the distance is `10 ** (DM / 5 + 1)` parsecs.

In [ ]:
w = lsdb.open_catalog(str(outBase))
print(w.npartitions, "partitions,", len(w.columns), "columns")
w.head(5)

In [ ]:
import glob
files = sorted(glob.glob(f"{outBase}/dataset/**/*.parquet", recursive=True))
cols = ["chi2min", "flags", "DM_quantile_median", "Mr_quantile_median", "FeH_quantile_median",
        "Ar_quantile_median"]
d = {c: [] for c in cols}
for f in files[::max(1, len(files) // 200)]:
    t = pq.read_table(f, columns=cols)
    for c in cols:
        d[c].append(t[c].to_numpy(zero_copy_only=False))
d = {c: np.concatenate(v) for c, v in d.items()}
flags = d["flags"].astype(int)
print(f"{len(flags):,} stars sampled")
print(f"  usable, flags & 3 == 0 : {100 * np.mean(flags & 3 == 0):.2f} %")
for bit, label in ((1, "poor fit or no answer"), (2, "two branches"), (4, "FeH at the grid end"),
                   (8, "A_r at the grid top"), (16, "a colour missing")):
    print(f"  bit {bit:2d} {label:22s}: {100 * np.mean((flags & bit) > 0):6.2f} %")
for c in ("DM_quantile_median", "Mr_quantile_median", "FeH_quantile_median", "Ar_quantile_median"):
    v = d[c].astype(float)
    print(f"  {c:24s} 5/50/95 %: " + "  ".join(f"{x:7.3f}" for x in np.nanpercentile(v, [5, 50, 95])))